In [13]:
import torch
from datasets import load_dataset
from torchtext.datasets import IMDB
from torch.utils.data import DataLoader
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
import torch.nn as nn
from collections import Counter


dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data  = dataset["test"]


print(Counter(train_data["label"]))  
print(Counter(test_data["label"]))   


def label_pipeline(label):
    return label  

Counter({0: 12500, 1: 12500})
Counter({0: 12500, 1: 12500})


In [14]:
tokenizer= get_tokenizer('basic_english')

from collections import defaultdict

vocab= defaultdict(lambda: 0)
counter= 1

for text in dataset['train']['text']:
    for token in text.split():
        if token not in vocab:
            vocab[token] = counter
            counter += 1

vocab_size= len(vocab) + 1


def text_pipeline(text,vocab=vocab):
    return [vocab[token] for token in tokenizer(text)]




In [15]:
class TextRNN(nn.Module):
    def __init__(self,vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()

        self.embedding= nn.Embedding(vocab_size, embed_dim)
        self.rnn= nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    
    def forward(self, text):
        embedded= self.embedding(text)
        out, hidden= self.rnn(embedded)
        out= self.fc(hidden[-1])

        return out 
    
vocab_size = len(vocab)
model = TextRNN(vocab_size=vocab_size, embed_dim=100, hidden_dim=256, output_dim=1)
print(model)


TextRNN(
  (embedding): Embedding(280617, 100)
  (rnn): RNN(100, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)


In [16]:
from torch.nn.utils.rnn import pad_sequence

def collate_batch(batch):

    text_list, label_list= [],[]
    max_len= 256

    for item in batch:
        label_list.append(item['label'])
        processed_text= torch.tensor(text_pipeline(item["text"],vocab)[:max_len], dtype=torch.int64)
        text_list.append(processed_text)


    text_list= pad_sequence(text_list, batch_first=True, padding_value=0)
    label_list= torch.tensor(label_list)

    return label_list, text_list

train_loader= DataLoader(train_data, batch_size=64, shuffle=True, collate_fn=collate_batch)


In [17]:
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model= model.to(device)

criterion= nn.BCEWithLogitsLoss()

optimizer= torch.optim.AdamW(model.parameters(), lr=0.001)

num_epoch=3

for epoch in range(num_epoch):

    model.train()
    running_loss=0

    for batch_idx, (label,text) in enumerate(train_loader):

        label= label.float().unsqueeze(1).to(device)
        text= text.to(device)

        output= model(text)
        loss= criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss+= loss.item()

        if batch_idx ==0:

          print(f"epoch {epoch+1}/{num_epoch} first batch loss : {loss.item()}")

        if (batch_idx + 1) % 100 ==0:
            print(f"epoch {epoch+1}/{num_epoch} | batch {batch_idx+1}/{len(train_loader)} | loss {loss.item()}")

    average_loss= running_loss/len(train_loader)
    print(f"epoch {epoch+1}/{num_epoch} |  average loss {average_loss}")



epoch 1/3 first batch loss : 0.7265715003013611
epoch 1/3 | batch 100/391 | loss 0.7128825187683105
epoch 1/3 | batch 200/391 | loss 0.6948180794715881
epoch 1/3 | batch 300/391 | loss 0.7056446075439453
epoch 1/3 |  average loss 0.6985990711490212
epoch 2/3 first batch loss : 0.6854471564292908
epoch 2/3 | batch 100/391 | loss 0.6715368032455444
epoch 2/3 | batch 200/391 | loss 0.6928303241729736
epoch 2/3 | batch 300/391 | loss 0.6891562938690186
epoch 2/3 |  average loss 0.6966271987351615
epoch 3/3 first batch loss : 0.6960441470146179
epoch 3/3 | batch 100/391 | loss 0.6906216144561768
epoch 3/3 | batch 200/391 | loss 0.6908893585205078
epoch 3/3 | batch 300/391 | loss 0.6846103072166443
epoch 3/3 |  average loss 0.6961252070448892


In [21]:
def predict_statement(text):

    with torch.no_grad():
        model.eval()

        text_tensor= torch.tensor(text_pipeline(text, vocab),dtype=torch.int64).unsqueeze(0).to(device)
        prediction= model(text_tensor).item()

        sentiment= 'positive' if prediction>0.5 else 'negative'

        return sentiment,prediction
    

reviews=[
    "The movie was really great, i enjoyed it a lot",
    "the movie was so bad i had to leave mid way",
    "the movie tried to be good but was eventually not worth it ",
    "this movie aint it "
]

for review in reviews:
    sentiment, prediction= predict_statement(review)
    print(f"Review: {review}")
    print(f"Sentiment: {sentiment} (score: {prediction:.4f})\n")





Review: The movie was really great, i enjoyed it a lot
Sentiment: negative (score: -0.3175)

Review: the movie was so bad i had to leave mid way
Sentiment: negative (score: -0.4271)

Review: the movie tried to be good but was eventually not worth it 
Sentiment: negative (score: -0.3822)

Review: this movie aint it 
Sentiment: negative (score: 0.3930)



In [19]:
import torchtext
print(torchtext.__version__)

0.18.0
